In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# 1. Tạo Spark session (tuned cho Jupyter)
spark = SparkSession.builder \
    .appName("ACN_Optimized") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "3g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "1g") \
    .getOrCreate()

print("="*60)
print("BƯỚC 1: ĐỌC DỮ LIỆU & PARSE DATETIME")
print("="*60)

# 2. Đọc json với schema defined (tránh inferSchema - nguyên nhân load chậm)
start_time = time.time()

# Định nghĩa schema chính xác (tăng tốc 2-3x)
schema = StructType([
    StructField("connectionTime", StringType(), True),
    StructField("disconnectTime", StringType(), True),
    StructField("doneChargingTime", StringType(), True),
    StructField("kWhDelivered", DoubleType(), True),
    StructField("sessionID", StringType(), True),
    StructField("siteID", StringType(), True),
    StructField("stationID", StringType(), True),
    StructField("userID", StringType(), True),
    # Bỏ qua các cột không cần
])

# Chỉ đọc các cột cần thiết
df_raw = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .json("hdfs://localhost:9000/ev-project/data/bronze/ev_sessions/caltech/*/*/*")  # Thay bằng path thật

print(f"✓ Load time with schema: {time.time()-start_time:.2f}s")
print(f"✓ Partitions: {df_raw.rdd.getNumPartitions()}")

# # 3. Xử lý datetime - KO dùng UDF, KO dùng to_timestamp nhiều lần
# start_time = time.time()

# df_clean = df_raw \
#     .withColumn("conn_ts", 
#                 to_timestamp(col("connectionTime"), "yyyy-MM-dd'T'HH:mm:ss.SSSXXX")) \
#     .withColumn("disc_ts", 
#                 to_timestamp(col("disconnectTime"), "yyyy-MM-dd'T'HH:mm:ss.SSSXXX")) \
#     .withColumn("done_ts", 
#                 to_timestamp(col("doneChargingTime"), "yyyy-MM-dd'T'HH:mm:ss.SSSXXX")) \
#     .drop("connectionTime", "disconnectTime", "doneChargingTime")  # Giải phóng memory

# # Thêm các cột datetime cơ bản
# df_clean = df_clean \
#     .withColumn("hour", hour(col("conn_ts"))) \
#     .withColumn("day_of_week", dayofweek(col("conn_ts")) - 1) \
#     .withColumn("month", month(col("conn_ts"))) \
#     .withColumn("year", year(col("conn_ts"))) \
#     .withColumn("date", to_date(col("conn_ts")))

# print(f"✓ DateTime processing: {time.time()-start_time:.2f}s")

# 4. Cache và kiểm tra memory
df_clean = df_clean.cache()
row_count = df_clean.count()  # Force cache

print(f"\n📊 RESULTS:")
print(f"Total sessions: {row_count:,}")
print(f"Memory usage: ~{df_clean.storageLevel.useMemory/1024/1024:.0f} MB")
print(f"Partitions: {df_clean.rdd.getNumPartitions()}")

# 5. Xem sample data
print("\n📋 SAMPLE DATA:")
df_clean.show(5, truncate=False)

# 6. Thống kê missing values
print("\n📊 MISSING VALUES:")
from pyspark.sql.functions import col, sum as spark_sum
missing_counts = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_clean.columns
])
missing_counts.show()

BƯỚC 1: ĐỌC DỮ LIỆU & PARSE DATETIME


✓ Load time with schema: 9.08s
✓ Partitions: 34


26/04/14 08:22:23 WARN CacheManager: Asked to cache already cached data.



📊 RESULTS:
Total sessions: 31,424
Memory usage: ~0 MB
Partitions: 34

📋 SAMPLE DATA:
+------------------+--------------------------------------+------+-----------+------+-------+-------+-------+----+-----------+-----+----+----+
|kWhDelivered      |sessionID                             |siteID|stationID  |userID|conn_ts|disc_ts|done_ts|hour|day_of_week|month|year|date|
+------------------+--------------------------------------+------+-----------+------+-------+-------+-------+----+-----------+-----+----+----+
|1.342             |2_39_139_28_2018-09-26 05:42:05.956498|0002  |2-39-139-28|NULL  |NULL   |NULL   |NULL   |NULL|NULL       |NULL |NULL|NULL|
|1.726             |2_39_91_437_2018-09-26 05:03:35.184064|0002  |2-39-91-437|NULL  |NULL   |NULL   |NULL   |NULL|NULL       |NULL |NULL|NULL|
|15.187641875756452|2_39_78_366_2018-09-26 04:35:26.259089|0002  |2-39-78-366|NULL  |NULL   |NULL   |NULL   |NULL|NULL       |NULL |NULL|NULL|
|2.375827359618296 |2_39_78_360_2018-09-26 04:34:23.6717

In [ ]:
# Fix 1: đọc data raw

In [61]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("ACN_Preprocessing") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.executor.memory", "3g") \
    .getOrCreate()

print("="*70)
print("STEP 1: LOAD RAW DATA")
print("="*70)

# Đọc toàn bộ dữ liệu thô (giữ nguyên các cột)
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .json("hdfs://localhost:9000/ev-project/data/bronze/ev_sessions/caltech/*/*/*")

print(f"Raw sessions: {df_raw.count():,}")
print(f"Raw columns: {df_raw.columns}")
df_raw.printSchema()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
STEP 1: LOAD RAW DATA


[Stage 12342:=================================================>   (32 + 2) / 34]

Raw sessions: 31,424
Raw columns: ['_batch_id', '_id', '_ingest_time', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs']
root
 |-- _batch_id: string (nullable = true)
 |-- _id: string (nullable = true)
 |-- _ingest_time: string (nullable = true)
 |-- clusterID: string (nullable = true)
 |-- connectionTime: string (nullable = true)
 |-- disconnectTime: string (nullable = true)
 |-- doneChargingTime: string (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- sessionID: string (nullable = true)
 |-- siteID: string (nullable = true)
 |-- spaceID: string (nullable = true)
 |-- stationID: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userID: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double 

In [ ]:
# Remove các cột id

In [50]:
print("="*70)
print("STEP 2: REMOVE ID COLUMNS")
print("="*70)

# Paper: "_id, sessionID, stationID, spaceID, siteID, clusterID, userID"
id_columns = ['_batch_id', '_id', '_ingest_time', 'sessionID', 
                        'stationID', 'spaceID', 'siteID', 'clusterID', 'userID']

# Kiểm tra xem có tồn tại không
existing_cols = [c for c in id_columns if c in df_raw.columns]
missing_cols = [c for c in id_columns if c not in df_raw.columns]

print(f"Removing: {existing_cols}")
if missing_cols:
    print(f"⚠️ Missing (already not in data): {missing_cols}")
    
df_step2 = df_raw.drop(*id_columns)

print(f"Removed columns: {id_columns}")
print(f"Remaining columns: {df_step2.columns}")
print(f"Row count: {df_step2.count():,}")
df_step2.printSchema()

STEP 2: REMOVE ID COLUMNS
Removing: ['_batch_id', '_id', '_ingest_time', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'userID']
Removed columns: ['_batch_id', '_id', '_ingest_time', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'userID']
Remaining columns: ['connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'timezone', 'userInputs']


[Stage 9107:==================================================>   (32 + 2) / 34]

Row count: 31,424
root
 |-- connectionTime: string (nullable = true)
 |-- disconnectTime: string (nullable = true)
 |-- doneChargingTime: string (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double (nullable = true)
 |    |    |-- milesRequested: long (nullable = true)
 |    |    |-- minutesAvailable: long (nullable = true)
 |    |    |-- modifiedAt: string (nullable = true)
 |    |    |-- paymentRequired: boolean (nullable = true)
 |    |    |-- requestedDeparture: string (nullable = true)
 |    |    |-- userID: long (nullable = true)



In [51]:
print("="*70)
print("STEP 3: CONVERT DATETIME TO TIMEZONE-NAIVE")
print("="*70)

# Kiểm tra format datetime hiện tại
print("📋 Raw datetime samples:")
df_step2.select("connectionTime", "disconnectTime", "doneChargingTime", "timezone").show(5, truncate=False)

# Paper: convert to timezone-naive format
# Dữ liệu có dạng: "2018-09-25 22:42:06-07:00"
# Dùng from_utc_timestamp để chuyển về UTC và bỏ timezone

df_step3 = df_step2 \
    .withColumn("conn_ts_with_tz", 
                to_timestamp(col("connectionTime"), "yyyy-MM-dd HH:mm:ssXXX")) \
    .withColumn("disc_ts_with_tz", 
                to_timestamp(col("disconnectTime"), "yyyy-MM-dd HH:mm:ssXXX")) \
    .withColumn("done_ts_with_tz", 
                to_timestamp(col("doneChargingTime"), "yyyy-MM-dd HH:mm:ssXXX"))

# Chuyển về UTC (bỏ timezone)
df_step3 = df_step3 \
    .withColumn("connectionTime_utc", 
                from_utc_timestamp(col("conn_ts_with_tz"), "UTC")) \
    .withColumn("disconnectTime_utc", 
                from_utc_timestamp(col("disc_ts_with_tz"), "UTC")) \
    .withColumn("doneChargingTime_utc", 
                from_utc_timestamp(col("done_ts_with_tz"), "UTC")) \
    .drop("connectionTime", "disconnectTime", "doneChargingTime", "timezone",
          "conn_ts_with_tz", "disc_ts_with_tz", "done_ts_with_tz")

print("\n✅ After conversion (timezone-naive UTC):")
df_step3.select("connectionTime_utc", "disconnectTime_utc", "doneChargingTime_utc").show(5, truncate=False)

# Kiểm tra null counts
null_counts = df_step3.select(
    sum(col("connectionTime_utc").isNull().cast("int")).alias("conn_null"),
    sum(col("disconnectTime_utc").isNull().cast("int")).alias("disc_null"),
    sum(col("doneChargingTime_utc").isNull().cast("int")).alias("done_null")
).collect()[0]

print(f"\n📊 Null counts after conversion:")
print(f"  connectionTime_utc null: {null_counts['conn_null']:,}")
print(f"  disconnectTime_utc null: {null_counts['disc_null']:,}")
print(f"  doneChargingTime_utc null: {null_counts['done_null']:,}")

# Kiểm tra time range
print("\n📅 Time range:")
df_step3.select(
    min("connectionTime_utc").alias("earliest"),
    max("connectionTime_utc").alias("latest")
).show()

print(f"\n✅ Remaining columns: {df_step3.columns}")
print(f"✅ Row count: {df_step3.count():,}")

# Bỏ userInputs (paper không dùng)
df_step3 = df_step3.drop("userInputs")
print(f"\n✅ Dropped 'userInputs' (not used in paper)")
print(f"✅ Final columns for next step: {df_step3.columns}")

STEP 3: CONVERT DATETIME TO TIMEZONE-NAIVE
📋 Raw datetime samples:
+-------------------------+-------------------------+-------------------------+-------------------+
|connectionTime           |disconnectTime           |doneChargingTime         |timezone           |
+-------------------------+-------------------------+-------------------------+-------------------+
|2018-09-25 22:42:06-07:00|2018-09-25 22:54:07-07:00|2018-09-25 22:54:03-07:00|America/Los_Angeles|
|2018-09-25 22:03:35-07:00|2018-09-25 22:18:53-07:00|2018-09-25 22:18:49-07:00|America/Los_Angeles|
|2018-09-25 21:35:26-07:00|2018-09-26 07:33:41-07:00|2018-09-26 07:01:30-07:00|America/Los_Angeles|
|2018-09-25 21:34:24-07:00|2018-09-25 22:26:26-07:00|2018-09-25 22:21:37-07:00|America/Los_Angeles|
|2018-09-25 20:36:28-07:00|2018-09-26 09:43:48-07:00|2018-09-25 22:45:03-07:00|America/Los_Angeles|
+-------------------------+-------------------------+-------------------------+-------------------+
only showing top 5 rows


✅ After


📊 Null counts after conversion:
  connectionTime_utc null: 0
  disconnectTime_utc null: 0
  doneChargingTime_utc null: 2,055

📅 Time range:


+-------------------+-------------------+
|           earliest|             latest|
+-------------------+-------------------+
|2018-04-25 11:08:04|2021-09-14 01:52:37|
+-------------------+-------------------+


✅ Remaining columns: ['kWhDelivered', 'userInputs', 'connectionTime_utc', 'disconnectTime_utc', 'doneChargingTime_utc']


[Stage 9118:=================================================>    (31 + 3) / 34]

✅ Row count: 31,424

✅ Dropped 'userInputs' (not used in paper)
✅ Final columns for next step: ['kWhDelivered', 'connectionTime_utc', 'disconnectTime_utc', 'doneChargingTime_utc']


In [52]:
print("="*70)
print("STEP 4: DURATION FEATURES (in hours)")
print("="*70)

from pyspark.sql.functions import unix_timestamp, col, log1p, when

# Paper: duration = disconnectTime - connectionTime (hours)
# Paper: charging_duration = doneChargingTime - connectionTime (hours)

df_step4 = df_step3 \
    .withColumn("duration", 
                (unix_timestamp(col("disconnectTime_utc")) - 
                 unix_timestamp(col("connectionTime_utc"))) / 3600) \
    .withColumn("charging_duration", 
                when(col("doneChargingTime_utc").isNotNull(),
                     (unix_timestamp(col("doneChargingTime_utc")) - 
                      unix_timestamp(col("connectionTime_utc"))) / 3600)
                .otherwise(None))

# Log transform of charging_duration (paper: charging_duration_log)
df_step4 = df_step4 \
    .withColumn("charging_duration_log", 
                when(col("charging_duration") > 0, log1p(col("charging_duration")))
                .otherwise(None))

print("📊 Duration statistics (hours):")
df_step4.select("duration", "charging_duration", "charging_duration_log").describe().show()

print("\n📋 Sample duration features (first 10 rows):")
df_step4.select(
    "connectionTime_utc", "disconnectTime_utc", "doneChargingTime_utc",
    "duration", "charging_duration", "charging_duration_log"
).show(10, truncate=False)

# Kiểm tra giá trị không hợp lệ
print("\n📊 Data quality checks:")
invalid_duration = df_step4.filter(col("duration") < 0).count()
invalid_charging = df_step4.filter(col("charging_duration") < 0).count()
print(f"  Negative duration: {invalid_duration:,}")
print(f"  Negative charging_duration: {invalid_charging:,}")

# Kiểm tra phân bố
print("\n📊 Duration distribution (hours):")
df_step4.select("duration").summary("min", "25%", "50%", "75%", "max").show()

print(f"\n✅ Remaining columns: {df_step4.columns}")
print(f"✅ Row count: {df_step4.count():,}")

STEP 4: DURATION FEATURES (in hours)
📊 Duration statistics (hours):


+-------+--------------------+-------------------+---------------------+
|summary|            duration|  charging_duration|charging_duration_log|
+-------+--------------------+-------------------+---------------------+
|  count|               31424|              29369|                29341|
|   mean|   5.652122086444898|  2.977543933323497|    1.197854743801607|
| stddev|  5.9636861445413585| 3.4296022505991215|   0.5716383654504641|
|    min|0.034444444444444444|-0.6894444444444444| 8.329863038918628E-4|
|    max|  245.26916666666668| 200.01583333333335|    5.303383677759315|
+-------+--------------------+-------------------+---------------------+


📋 Sample duration features (first 10 rows):
+-------------------+-------------------+--------------------+-------------------+-------------------+---------------------+
|connectionTime_utc |disconnectTime_utc |doneChargingTime_utc|duration           |charging_duration  |charging_duration_log|
+-------------------+-------------------+------

  Negative duration: 0
  Negative charging_duration: 26

📊 Duration distribution (hours):


+-------+--------------------+
|summary|            duration|
+-------+--------------------+
|    min|0.034444444444444444|
|    25%|  1.8652777777777778|
|    50%|                4.73|
|    75%|   8.466111111111111|
|    max|  245.26916666666668|
+-------+--------------------+


✅ Remaining columns: ['kWhDelivered', 'connectionTime_utc', 'disconnectTime_utc', 'doneChargingTime_utc', 'duration', 'charging_duration', 'charging_duration_log']


[Stage 9134:==================================================>   (32 + 2) / 34]

✅ Row count: 31,424


In [53]:
print("="*70)
print("STEP 5: TEMPORAL FEATURES")
print("="*70)

from pyspark.sql.functions import hour, dayofweek, month, year, dayofyear, weekofyear
from pyspark.sql.functions import sin, cos, pi, when

df_step5 = df_step4

# Basic time features
df_step5 = df_step5 \
    .withColumn("hour", hour(col("connectionTime_utc"))) \
    .withColumn("day_of_week", dayofweek(col("connectionTime_utc")) - 1) \
    .withColumn("month", month(col("connectionTime_utc"))) \
    .withColumn("year", year(col("connectionTime_utc"))) \
    .withColumn("day_of_year", dayofyear(col("connectionTime_utc"))) \
    .withColumn("week_of_year", weekofyear(col("connectionTime_utc")))

# Season (paper: 0=Winter,1=Spring,2=Summer,3=Fall)
df_step5 = df_step5.withColumn("season",
    when(col("month").isin([12, 1, 2]), 0)
    .when(col("month").isin([3, 4, 5]), 1)
    .when(col("month").isin([6, 7, 8]), 2)
    .otherwise(3)  # 9,10,11 = Fall
)

# is_weekend (paper: 1 for Saturday/Sunday)
df_step5 = df_step5.withColumn("is_weekend", 
    when(col("day_of_week") >= 5, 1).otherwise(0))

# is_holiday (paper: 1 for December/January)
df_step5 = df_step5.withColumn("is_holiday",
    when(col("month").isin([12, 1]), 1).otherwise(0))

# Cyclical features for hour
df_step5 = df_step5 \
    .withColumn("hour_sin", sin(2 * pi() * col("hour") / 24)) \
    .withColumn("hour_cos", cos(2 * pi() * col("hour") / 24))

print("✅ Temporal features added:")
print(f"  - hour: 0-23")
print(f"  - day_of_week: 0-6 (0=Monday)")
print(f"  - month: 1-12")
print(f"  - year: {df_step5.select('year').distinct().orderBy('year').collect()}")
print(f"  - day_of_year: 1-365/366")
print(f"  - week_of_year: 1-52/53")
print(f"  - season: 0=Winter,1=Spring,2=Summer,3=Fall")
print(f"  - is_weekend: 0/1")
print(f"  - is_holiday: 0/1 (1 for Dec/Jan)")
print(f"  - hour_sin, hour_cos: circular encoding")

# Sample data
print("\n📋 Sample temporal features (first 10 rows):")
df_step5.select(
    "connectionTime_utc", "hour", "day_of_week", "month", "year",
    "day_of_year", "week_of_year", "season", "is_weekend", "is_holiday",
    "hour_sin", "hour_cos"
).show(10, truncate=False)

# Verify is_holiday logic
print("\n📊 is_holiday distribution:")
df_step5.groupBy("month", "is_holiday").count().orderBy("month").show(13)

print(f"\n✅ Remaining columns: {len(df_step5.columns)} columns")
print(f"✅ Row count: {df_step5.count():,}")

STEP 5: TEMPORAL FEATURES
✅ Temporal features added:
  - hour: 0-23
  - day_of_week: 0-6 (0=Monday)
  - month: 1-12


  - year: [Row(year=2018), Row(year=2019), Row(year=2020), Row(year=2021)]
  - day_of_year: 1-365/366
  - week_of_year: 1-52/53
  - season: 0=Winter,1=Spring,2=Summer,3=Fall
  - is_weekend: 0/1
  - is_holiday: 0/1 (1 for Dec/Jan)
  - hour_sin, hour_cos: circular encoding

📋 Sample temporal features (first 10 rows):
+-------------------+----+-----------+-----+----+-----------+------------+------+----------+----------+-------------------+-------------------+
|connectionTime_utc |hour|day_of_week|month|year|day_of_year|week_of_year|season|is_weekend|is_holiday|hour_sin           |hour_cos           |
+-------------------+----+-----------+-----+----+-----------+------------+------+----------+----------+-------------------+-------------------+
|2018-09-26 05:42:06|5   |3          |9    |2018|269        |39          |3     |0         |0         |0.9659258262890683 |0.25881904510252074|
|2018-09-26 05:03:35|5   |3          |9    |2018|269        |39          |3     |0         |0         |0.96

+-----+----------+-----+
|month|is_holiday|count|
+-----+----------+-----+
|    1|         1| 1983|
|    2|         0| 1964|
|    3|         0| 1594|
|    4|         0| 1615|
|    5|         0| 3099|
|    6|         0| 3216|
|    7|         0| 3371|
|    8|         0| 3901|
|    9|         0| 3439|
|   10|         0| 3338|
|   11|         0| 2176|
|   12|         1| 1728|
+-----+----------+-----+


✅ Remaining columns: 18 columns


[Stage 9149:==============================================>       (29 + 4) / 34]

✅ Row count: 31,424


In [54]:
print("="*70)
print("STEP 6: LAG FEATURES (global time order, NO stationID)")
print("="*70)

from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

# Paper: lag features on charging_duration_log
# Sắp xếp theo thời gian TOÀN CỤC, không partition theo station

window_lag_global = Window.orderBy("connectionTime_utc")

# Tạo lag features (1, 2, 3 periods behind)
df_step6 = df_step5 \
    .withColumn("lag_1_log", lag("charging_duration_log", 1).over(window_lag_global)) \
    .withColumn("lag_2_log", lag("charging_duration_log", 2).over(window_lag_global)) \
    .withColumn("lag_3_log", lag("charging_duration_log", 3).over(window_lag_global))

print("✅ Lag features created (global time order):")
print("  - lag_1_log: charging_duration_log của session trước đó")
print("  - lag_2_log: charging_duration_log của 2 session trước")
print("  - lag_3_log: charging_duration_log của 3 session trước")
print("  ⚠️ NO partition by stationID (paper doesn't mention it)")

# Kiểm tra kết quả
print("\n📋 Sample lag features (first 20 rows):")
df_step6.select(
    "connectionTime_utc", "charging_duration_log",
    "lag_1_log", "lag_2_log", "lag_3_log"
).show(20, truncate=False)

# Kiểm tra null counts (chỉ 3 sessions đầu tiên bị null)
null_counts = df_step6.select(
    sum(col("lag_1_log").isNull().cast("int")).alias("lag1_null"),
    sum(col("lag_2_log").isNull().cast("int")).alias("lag2_null"),
    sum(col("lag_3_log").isNull().cast("int")).alias("lag3_null")
).collect()[0]

print(f"\n📊 Null counts:")
print(f"  lag_1_log null: {null_counts['lag1_null']:,} (first session only)")
print(f"  lag_2_log null: {null_counts['lag2_null']:,} (first 2 sessions)")
print(f"  lag_3_log null: {null_counts['lag3_null']:,} (first 3 sessions)")

print(f"\n✅ Remaining columns: {len(df_step6.columns)} columns")
print(f"✅ Row count: {df_step6.count():,}")

STEP 6: LAG FEATURES (global time order, NO stationID)
✅ Lag features created (global time order):
  - lag_1_log: charging_duration_log của session trước đó
  - lag_2_log: charging_duration_log của 2 session trước
  - lag_3_log: charging_duration_log của 3 session trước
  ⚠️ NO partition by stationID (paper doesn't mention it)

📋 Sample lag features (first 20 rows):


26/04/14 10:26:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:26:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:26:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:26:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:26:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

+-------------------+---------------------+------------------+------------------+------------------+
|connectionTime_utc |charging_duration_log|lag_1_log         |lag_2_log         |lag_3_log         |
+-------------------+---------------------+------------------+------------------+------------------+
|2018-04-25 11:08:04|1.168863627212368    |NULL              |NULL              |NULL              |
|2018-04-25 13:45:10|1.3824676039712638   |1.168863627212368 |NULL              |NULL              |
|2018-04-25 13:45:50|0.7411433788282008   |1.3824676039712638|1.168863627212368 |NULL              |
|2018-04-25 14:37:06|0.9046678920461622   |0.7411433788282008|1.3824676039712638|1.168863627212368 |
|2018-04-25 14:40:34|1.38601654475472     |0.9046678920461622|0.7411433788282008|1.3824676039712638|
|2018-04-25 14:43:50|0.9467121608674877   |1.38601654475472  |0.9046678920461622|0.7411433788282008|
|2018-04-25 14:47:42|1.5404450409471488   |0.9467121608674877|1.38601654475472  |0.90466789

26/04/14 10:26:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:26:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                


📊 Null counts:
  lag_1_log null: 2,083 (first session only)
  lag_2_log null: 2,084 (first 2 sessions)
  lag_3_log null: 2,084 (first 3 sessions)

✅ Remaining columns: 21 columns


[Stage 9158:====================================================> (33 + 1) / 34]

✅ Row count: 31,424


In [55]:
print("="*70)
print("STEP 7: ROLLING MEAN FEATURES")
print("="*70)

from pyspark.sql.window import Window
from pyspark.sql.functions import avg, col

# Paper: rolling_mean_3_log và rolling_mean_5_log
# Dựa trên charging_duration_log
# Window: 3 và 5 periods trước đó (không tính period hiện tại)

window_roll3 = Window.orderBy("connectionTime_utc").rowsBetween(-3, -1)
window_roll5 = Window.orderBy("connectionTime_utc").rowsBetween(-5, -1)

df_step7 = df_step6 \
    .withColumn("rolling_mean_3_log", avg("charging_duration_log").over(window_roll3)) \
    .withColumn("rolling_mean_5_log", avg("charging_duration_log").over(window_roll5))

print("✅ Rolling mean features created:")
print("  - rolling_mean_3_log: average of charging_duration_log of 3 previous sessions")
print("  - rolling_mean_5_log: average of charging_duration_log of 5 previous sessions")

# Kiểm tra kết quả
print("\n📋 Sample rolling mean features (first 20 rows):")
df_step7.select(
    "connectionTime_utc", "charging_duration_log",
    "rolling_mean_3_log", "rolling_mean_5_log"
).show(20, truncate=False)

# Kiểm tra null counts (các session đầu tiên sẽ bị null)
null_counts = df_step7.select(
    sum(col("rolling_mean_3_log").isNull().cast("int")).alias("roll3_null"),
    sum(col("rolling_mean_5_log").isNull().cast("int")).alias("roll5_null")
).collect()[0]

print(f"\n📊 Null counts:")
print(f"  rolling_mean_3_log null: {null_counts['roll3_null']:,} (first 3 sessions)")
print(f"  rolling_mean_5_log null: {null_counts['roll5_null']:,} (first 5 sessions)")

# Thống kê rolling mean
print("\n📊 Rolling mean statistics (non-null values):")
df_step7.filter(col("rolling_mean_3_log").isNotNull()).select(
    "rolling_mean_3_log", "rolling_mean_5_log"
).describe().show()

print(f"\n✅ Remaining columns: {len(df_step7.columns)} columns")
print(f"✅ Row count: {df_step7.count():,}")

# Hiển thị danh sách columns hiện tại
print("\n📋 Current columns:")
for i, col_name in enumerate(df_step7.columns):
    print(f"  {i+1:2}. {col_name}")

STEP 7: ROLLING MEAN FEATURES
✅ Rolling mean features created:
  - rolling_mean_3_log: average of charging_duration_log of 3 previous sessions
  - rolling_mean_5_log: average of charging_duration_log of 5 previous sessions

📋 Sample rolling mean features (first 20 rows):


26/04/14 10:27:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

+-------------------+---------------------+------------------+------------------+
|connectionTime_utc |charging_duration_log|rolling_mean_3_log|rolling_mean_5_log|
+-------------------+---------------------+------------------+------------------+
|2018-04-25 11:08:04|1.168863627212368    |NULL              |NULL              |
|2018-04-25 13:45:10|1.3824676039712638   |1.168863627212368 |1.168863627212368 |
|2018-04-25 13:45:50|0.7411433788282008   |1.2756656155918158|1.2756656155918158|
|2018-04-25 14:37:06|0.9046678920461622   |1.0974915366706108|1.0974915366706108|
|2018-04-25 14:40:34|1.38601654475472     |1.009426291615209 |1.0492856255144987|
|2018-04-25 14:43:50|0.9467121608674877   |1.010609271876361 |1.116631809362543 |
|2018-04-25 14:47:42|1.5404450409471488   |1.07913219922279  |1.072201516093567 |
|2018-04-25 14:58:25|1.0418459548175008   |1.2910579155231188|1.103797003488744 |
|2018-04-25 15:10:52|0.6683990108707513   |1.1763343855440456|1.163937518686604 |
|2018-04-25 15:1

26/04/14 10:27:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                


📊 Null counts:
  rolling_mean_3_log null: 1,058 (first 3 sessions)
  rolling_mean_5_log null: 856 (first 5 sessions)

📊 Rolling mean statistics (non-null values):


26/04/14 10:27:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:27:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

+-------+-------------------+-------------------+
|summary| rolling_mean_3_log| rolling_mean_5_log|
+-------+-------------------+-------------------+
|  count|              30366|              30366|
|   mean|  1.189184244175747| 1.1895577362653835|
| stddev|0.37868509320069643| 0.3218848866107085|
|    min|0.08872287450480301|0.18324705431489752|
|    max|  3.506716218119916|  3.506716218119916|
+-------+-------------------+-------------------+


✅ Remaining columns: 23 columns


[Stage 9170:==============================================>       (29 + 4) / 34]

✅ Row count: 31,424

📋 Current columns:
   1. kWhDelivered
   2. connectionTime_utc
   3. disconnectTime_utc
   4. doneChargingTime_utc
   5. duration
   6. charging_duration
   7. charging_duration_log
   8. hour
   9. day_of_week
  10. month
  11. year
  12. day_of_year
  13. week_of_year
  14. season
  15. is_weekend
  16. is_holiday
  17. hour_sin
  18. hour_cos
  19. lag_1_log
  20. lag_2_log
  21. lag_3_log
  22. rolling_mean_3_log
  23. rolling_mean_5_log


In [56]:
print("="*70)
print("STEP 8: OUTLIER REMOVAL USING IQR")
print("="*70)

from pyspark.sql.functions import col, expr

# Paper: "Outlier management employed the Interquartile Range (IQR) method, 
# calculating Q1 and Q3 for duration, charging_duration, and kWhDelivered"

def remove_outliers_iqr(df, column_name, verbose=True):
    """Remove outliers using IQR method"""
    
    # Tính quantiles
    quantiles = df.approxQuantile(column_name, [0.25, 0.75], 0.01)
    q1 = quantiles[0]
    q3 = quantiles[1]
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    if verbose:
        print(f"\n  {column_name}:")
        print(f"    Q1 = {q1:.4f}, Q3 = {q3:.4f}, IQR = {iqr:.4f}")
        print(f"    Lower bound = {lower_bound:.4f}, Upper bound = {upper_bound:.4f}")
    
    # Đếm outliers
    before_count = df.count()
    outliers = df.filter((col(column_name) < lower_bound) | (col(column_name) > upper_bound))
    outlier_count = outliers.count()
    
    if verbose:
        print(f"    Outliers found: {outlier_count:,} ({100*outlier_count/before_count:.2f}%)")
    
    # Lọc outliers
    df_clean = df.filter((col(column_name) >= lower_bound) & (col(column_name) <= upper_bound))
    
    return df_clean, outlier_count

# Tạo bản copy để xử lý outlier
df_step8 = df_step7

print("\n📊 Removing outliers from DURATION (hours):")
df_step8, dur_outliers = remove_outliers_iqr(df_step8, "duration")

print("\n📊 Removing outliers from CHARGING_DURATION (hours):")
df_step8, charge_outliers = remove_outliers_iqr(df_step8, "charging_duration")

print("\n📊 Removing outliers from KWHDELIVERED (target):")
df_step8, kwh_outliers = remove_outliers_iqr(df_step8, "kWhDelivered")

# Tổng kết
print("\n" + "="*50)
print("📊 OUTLIER REMOVAL SUMMARY:")
print(f"  duration outliers removed: {dur_outliers:,}")
print(f"  charging_duration outliers removed: {charge_outliers:,}")
print(f"  kWhDelivered outliers removed: {kwh_outliers:,}")

print(f"\n✅ Before outlier removal: {df_step7.count():,} rows")
print(f"✅ After outlier removal: {df_step8.count():,} rows")
print(f"✅ Total rows removed: {df_step7.count() - df_step8.count():,}")

# Kiểm tra lại statistics sau khi xử lý
print("\n📊 STATISTICS AFTER OUTLIER REMOVAL:")
df_step8.select("duration", "charging_duration", "kWhDelivered").describe().show()

# Kiểm tra còn giá trị âm không
print("\n📊 DATA QUALITY CHECK AFTER OUTLIER REMOVAL:")
negative_check = df_step8.select(
    sum((col("duration") < 0).cast("int")).alias("neg_duration"),
    sum((col("charging_duration") < 0).cast("int")).alias("neg_charging"),
    sum((col("kWhDelivered") < 0).cast("int")).alias("neg_kwh")
).collect()[0]

print(f"  Negative duration: {negative_check['neg_duration']:,}")
print(f"  Negative charging_duration: {negative_check['neg_charging']:,}")
print(f"  Negative kWhDelivered: {negative_check['neg_kwh']:,}")

print(f"\n✅ Remaining columns: {len(df_step8.columns)} columns")
print(f"✅ Row count: {df_step8.count():,}")

STEP 8: OUTLIER REMOVAL USING IQR

📊 Removing outliers from DURATION (hours):



  duration:
    Q1 = 1.8489, Q3 = 8.3992, IQR = 6.5503
    Lower bound = -7.9765, Upper bound = 18.2246


    Outliers found: 441 (1.40%)

📊 Removing outliers from CHARGING_DURATION (hours):



  charging_duration:
    Q1 = 1.1503, Q3 = 3.7442, IQR = 2.5939
    Lower bound = -2.7406, Upper bound = 7.6350


    Outliers found: 1,949 (6.29%)

📊 Removing outliers from KWHDELIVERED (target):


...

✅ Total rows removed: 5,329

📊 STATISTICS AFTER OUTLIER REMOVAL:


+-------+-------------------+-------------------+-----------------+
|summary|           duration|  charging_duration|     kWhDelivered|
+-------+-------------------+-------------------+-----------------+
|  count|              26095|              26095|            26095|
|   mean|  4.969309659151392|    2.2829554725256|7.254670557563193|
| stddev| 3.4807017727368144| 1.6318368518778215| 5.14837470672348|
|    min|0.07777777777777778|-0.6894444444444444|            0.501|
|    max|  18.17111111111111| 7.6338888888888885|           24.417|
+-------+-------------------+-------------------+-----------------+


📊 DATA QUALITY CHECK AFTER OUTLIER REMOVAL:


  Negative duration: 0
  Negative charging_duration: 23
  Negative kWhDelivered: 0

✅ Remaining columns: 23 columns


[Stage 9215:==============================================>       (29 + 4) / 34]

✅ Row count: 26,095


In [57]:
print("="*70)
print("STEP 9: REMOVE MISSING VALUES & LOG TRANSFORM TARGET")
print("="*70)

from pyspark.sql.functions import col, sum as spark_sum, log1p
import numpy as np

# Paper: "Missing values, arising from feature engineering or lags, were removed"
# Paper: "log transform for the target to improve model fit (skewness = 1.09)"

# Bước 9.1: Kiểm tra missing values trước khi xóa
print("\n📊 MISSING VALUES BEFORE REMOVAL:")
missing_before = df_step8.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_step8.columns if c not in ['connectionTime_utc', 'disconnectTime_utc', 'doneChargingTime_utc']
])
missing_before.show(vertical=True, truncate=False)

# Bước 9.2: Xóa các row có missing values
print("\n🗑️ REMOVING MISSING VALUES...")

# Các cột quan trọng không được phép có null
critical_cols = ['kWhDelivered', 'duration', 'charging_duration_log', 
                 'lag_1_log', 'lag_2_log', 'lag_3_log',
                 'rolling_mean_3_log', 'rolling_mean_5_log']

before_count = df_step8.count()
df_step9 = df_step8

for col_name in critical_cols:
    df_step9 = df_step9.filter(col(col_name).isNotNull())
    after_count = df_step9.count()
    if after_count < before_count:
        print(f"  Removed {before_count - after_count:,} rows with null in '{col_name}'")
        before_count = after_count

# Xóa các row có charging_duration âm (còn 23 cái)
df_step9 = df_step9.filter(col("charging_duration") >= 0)
print(f"  Removed {df_step8.count() - df_step9.count() - (df_step8.count() - before_count):,} rows with negative charging_duration")

print(f"\n✅ After removing missing values: {df_step9.count():,} rows")
print(f"✅ Total rows removed: {df_step8.count() - df_step9.count():,}")

# Bước 9.3: Kiểm tra lại missing values
print("\n📊 REMAINING NULLS (should be 0 for all columns):")
missing_after = df_step9.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in critical_cols
])
missing_after.show(vertical=True, truncate=False)

# Bước 9.4: Log transform target variable (kWhDelivered)
print("\n📊 LOG TRANSFORM TARGET VARIABLE:")

# Kiểm tra skewness trước khi log transform
sample_pdf = df_step9.select("kWhDelivered").sample(0.1, seed=42).toPandas()
skewness_before = sample_pdf["kWhDelivered"].skew()
print(f"  Skewness before log transform: {skewness_before:.4f}")
print(f"  Paper reported skewness: 1.09")

# Log transform: log1p = log(1 + x)
df_step9 = df_step9.withColumn("kWhDelivered_log", log1p(col("kWhDelivered")))

# Kiểm tra skewness sau log transform
sample_pdf_after = df_step9.select("kWhDelivered_log").sample(0.1, seed=42).toPandas()
skewness_after = sample_pdf_after["kWhDelivered_log"].skew()
print(f"  Skewness after log transform: {skewness_after:.4f}")
print(f"  Improvement: {skewness_before - skewness_after:.4f}")

# Thống kê target
print("\n📊 TARGET STATISTICS:")
df_step9.select("kWhDelivered", "kWhDelivered_log").describe().show()

# Bước 9.5: Chỉ giữ các features đúng theo paper (17 features + target)
print("\n📋 SELECTING FINAL FEATURES (17 features + target):")

features_paper = [
    'hour', 'day_of_week', 'month', 'season',
    'duration', 'charging_duration', 'charging_duration_log',
    'hour_sin', 'hour_cos', 'day_of_year', 'week_of_year', 'is_holiday',
    'lag_1_log', 'lag_2_log', 'lag_3_log',
    'rolling_mean_3_log', 'rolling_mean_5_log'
]

# Kiểm tra xem có đủ features không
missing_features = [f for f in features_paper if f not in df_step9.columns]
if missing_features:
    print(f"  ⚠️ Missing features: {missing_features}")
else:
    print(f"  ✅ All 17 features present")

# Chọn final columns
final_columns = features_paper + ['kWhDelivered_log', 'connectionTime_utc']
df_final = df_step9.select(*final_columns)

print(f"\n✅ Final dataset: {df_final.count():,} rows, {len(df_final.columns)} columns")
print(f"✅ Features: {len(features_paper)}")
print(f"✅ Target: kWhDelivered_log")

# So sánh với paper
print(f"\n📊 COMPARISON WITH PAPER:")
print(f"  Paper's cleaned sessions: 14,496")
print(f"  Our cleaned sessions: {df_final.count():,}")
print(f"  Difference: {df_final.count() - 14496:,}")

# Hiển thị sample
print("\n📋 FINAL DATASET SAMPLE (first 5 rows):")
df_final.show(5, truncate=False)

print(f"\n✅ Final columns: {df_final.columns}")

STEP 9: REMOVE MISSING VALUES & LOG TRANSFORM TARGET

📊 MISSING VALUES BEFORE REMOVAL:


26/04/14 10:28:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:28:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:28:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:28:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

-RECORD 0--------------------
 kWhDelivered          | 0   
 duration              | 0   
 charging_duration     | 0   
 charging_duration_log | 25  
 hour                  | 0   
 day_of_week           | 0   
 month                 | 0   
 year                  | 0   
 day_of_year           | 0   
 week_of_year          | 0   
 season                | 0   
 is_weekend            | 0   
 is_holiday            | 0   
 hour_sin              | 0   
 hour_cos              | 0   
 lag_1_log             | 692 
 lag_2_log             | 675 
 lag_3_log             | 703 
 rolling_mean_3_log    | 117 
 rolling_mean_5_log    | 33  


🗑️ REMOVING MISSING VALUES...


26/04/14 10:28:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:28:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


  Removed 25 rows with null in 'charging_duration_log'


...

  Our cleaned sessions: 24,687


26/04/14 10:29:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:29:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:29:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:29:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:29:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:29:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 1

  Difference: 10,191

📋 FINAL DATASET SAMPLE (first 5 rows):


26/04/14 10:29:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:29:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

+----+-----------+-----+------+------------------+------------------+---------------------+-------------------+-------------------+-----------+------------+----------+------------------+------------------+------------------+------------------+------------------+-----------------+-------------------+
|hour|day_of_week|month|season|duration          |charging_duration |charging_duration_log|hour_sin           |hour_cos           |day_of_year|week_of_year|is_holiday|lag_1_log         |lag_2_log         |lag_3_log         |rolling_mean_3_log|rolling_mean_5_log|kWhDelivered_log |connectionTime_utc |
+----+-----------+-----+------+------------------+------------------+---------------------+-------------------+-------------------+-----------+------------+----------+------------------+------------------+------------------+------------------+------------------+-----------------+-------------------+
|14  |3          |4    |1     |9.307777777777778 |1.471111111111111 |0.9046678920461622   |-0.499

In [44]:
print("="*70)
print("STEP 10: TRAIN/TEST SPLIT (80/20, SEED=42, NO SHUFFLE)")
print("="*70)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, rand

# Paper: "randomly split into 80% training and 20% testing without shuffling"
# "without shuffling" = không xáo trộn thứ tự thời gian
# "randomly split" = random sampling với seed=42

# Cách 1: Random sample (đúng với paper)
print("\n📌 Method 1: Random split with seed=42 (as paper describes)")

total_rows = df_final.count()
train_size = int(total_rows * 0.8)
test_size = total_rows - train_size

print(f"  Total: {total_rows:,}")
print(f"  Train (80%): {train_size:,}")
print(f"  Test (20%): {test_size:,}")

# Random split với seed=42
train_df_random = df_final.sample(fraction=0.8, seed=42)
test_df_random = df_final.subtract(train_df_random)

print(f"\n✅ Random split (seed=42):")
print(f"  Train: {train_df_random.count():,}")
print(f"  Test: {test_df_random.count():,}")

# Cách 2: Time-based split (temporal consistency)
print("\n📌 Method 2: Time-based split (no shuffle, temporal order)")

df_ordered = df_final.orderBy("connectionTime_utc")
window_order = Window.orderBy("connectionTime_utc")
df_with_index = df_ordered.withColumn("row_num", row_number().over(window_order))

train_df_time = df_with_index.filter(col("row_num") <= train_size).drop("row_num")
test_df_time = df_with_index.filter(col("row_num") > train_size).drop("row_num")

print(f"\n✅ Time-based split:")
print(f"  Train: {train_df_time.count():,}")
print(f"  Test: {test_df_time.count():,}")

# Kiểm tra overlap
train_max = train_df_time.select(max("connectionTime_utc")).collect()[0][0]
test_min = test_df_time.select(min("connectionTime_utc")).collect()[0][0]
print(f"\n📅 Temporal check:")
print(f"  Train max time: {train_max}")
print(f"  Test min time: {test_min}")
print(f"  No overlap: {train_max < test_min}")

# Paper's choice: Which one to use?
print("\n📌 PAPER'S METHOD:")
print("  'randomly split into 80% training and 20% testing without shuffling'")
print("  → Random split with seed=42 (Method 1)")

# Chọn method 1 (random split) theo paper
train_df = train_df_random
test_df = test_df_random

# Drop connectionTime_utc (không dùng cho training)
train_df = train_df.drop("connectionTime_utc")
test_df = test_df.drop("connectionTime_utc")

print(f"\n✅ FINAL TRAIN/TEST SETS (paper's method):")
print(f"  Train features: {len(train_df.columns)}")
print(f"  Test features: {len(test_df.columns)}")
print(f"  Train target: kWhDelivered_log")
print(f"  Test target: kWhDelivered_log")

# Save final datasets
print("\n💾 SAVING FINAL DATASETS...")
train_df.write.mode("overwrite").parquet("final_data/train_paper")
test_df.write.mode("overwrite").parquet("final_data/test_paper")

print("✅ Saved to: final_data/train_paper and final_data/test_paper")

# Verify no data leakage
print("\n🔍 DATA LEAKAGE CHECK:")
train_ids = train_df.select("kWhDelivered_log").collect()
test_ids = test_df.select("kWhDelivered_log").collect()
print(f"  No identical rows guarantee: random split with seed=42")

print("\n" + "="*70)
print("✅✅✅ PREPROCESSING COMPLETE! ✅✅✅")
print("="*70)
print(f"\n📊 FINAL SUMMARY:")
print(f"  Original sessions: 31,424")
print(f"  Clean sessions: {total_rows:,}")
print(f"  Train sessions: {train_df.count():,}")
print(f"  Test sessions: {test_df.count():,}")
print(f"  Features: {len(features_paper)}")
print(f"  Target: kWhDelivered_log")

STEP 10: TRAIN/TEST SPLIT (80/20, SEED=42, NO SHUFFLE)

📌 Method 1: Random split with seed=42 (as paper describes)


26/04/14 10:03:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:03:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

  Total: 24,687
  Train (80%): 19,749
  Test (20%): 4,938

✅ Random split (seed=42):


26/04/14 10:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:03:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:03:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

  Train: 19,777


...

  No identical rows guarantee: random split with seed=42

✅✅✅ PREPROCESSING COMPLETE! ✅✅✅

📊 FINAL SUMMARY:
  Original sessions: 31,424
  Clean sessions: 24,687


26/04/14 10:04:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:04:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:04:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:04:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

  Train sessions: 19,777


26/04/14 10:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 1

  Test sessions: 4,910
  Features: 17
  Target: kWhDelivered_log


In [46]:
print("="*70)
print("TRAIN GBT REGRESSOR (Spark ML)")
print("="*70)

from pyspark.ml.regression import GBTRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Features (17 features)
feature_cols = [
    'hour', 'day_of_week', 'month', 'season',
    'duration', 'charging_duration', 'charging_duration_log',
    'hour_sin', 'hour_cos', 'day_of_year', 'week_of_year', 'is_holiday',
    'lag_1_log', 'lag_2_log', 'lag_3_log',
    'rolling_mean_3_log', 'rolling_mean_5_log'
]

# Target
target_col = 'kWhDelivered_log'

# Pipeline
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
gbt = GBTRegressor(featuresCol="scaled_features", labelCol=target_col, maxDepth=10, maxIter=100, seed=42)

pipeline = Pipeline(stages=[assembler, scaler, gbt])

# Train
model = pipeline.fit(train_df)

# Predict
predictions = model.transform(test_df)

# Evaluate
evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
rmse_log = evaluator.evaluate(predictions)

print(f"RMSE (log scale): {rmse_log:.4f}")
print(f"RMSE (kWh): {np.expm1(rmse_log):.4f}")

TRAIN GBT REGRESSOR (Spark ML)


26/04/14 10:07:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:07:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:07:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:07:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:07:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:07:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 1

RMSE (log scale): 0.5011
RMSE (kWh): 0.6505


In [47]:
print("="*70)
print("DIAGNOSTIC: CHECK PREDICTIONS")
print("="*70)

from pyspark.sql.functions import col, exp, avg, sqrt, pow, abs

# Lấy sample predictions
predictions.select("kWhDelivered_log", "prediction").show(10, truncate=False)

# Chuyển về original scale
predictions_orig = predictions \
    .withColumn("actual_kwh", exp(col("kWhDelivered_log")) - 1) \
    .withColumn("pred_kwh", exp(col("prediction")) - 1)

print("\n📊 On original scale (kWh):")
predictions_orig.select("actual_kwh", "pred_kwh").show(10, truncate=False)

# Tính metrics trên original scale
metrics_orig = predictions_orig.select(
    avg(abs(col("actual_kwh") - col("pred_kwh"))).alias("mae"),
    sqrt(avg(pow(col("actual_kwh") - col("pred_kwh"), 2))).alias("rmse")
).collect()[0]

print(f"\n📊 METRICS ON ORIGINAL SCALE (kWh):")
print(f"  MAE: {metrics_orig['mae']:.4f} kWh")
print(f"  RMSE: {metrics_orig['rmse']:.4f} kWh")
print(f"\n  Paper's XGBoost MAE: 2.6697 kWh")
print(f"  Paper's XGBoost RMSE: 3.9451 kWh")

# So sánh
if metrics_orig['rmse'] < 1.0:
    print("\n⚠️ WARNING: Your RMSE is too low compared to paper!")
    print("   Possible issues:")
    print("   1. Data leakage - test set may be too similar to train")
    print("   2. Different dataset size (24,687 vs 14,496)")
    print("   3. Target variable mismatch")

DIAGNOSTIC: CHECK PREDICTIONS


26/04/14 10:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 1

+------------------+------------------+
|kWhDelivered_log  |prediction        |
+------------------+------------------+
|2.187174241482718 |1.916990564772657 |
|1.235180731450129 |2.3093949914013634|
|1.4466836611326745|0.9468232220996748|
|1.6180011430945882|1.9091136664024986|
|2.3564101796889534|1.9048496645361757|
|1.7134375024525852|3.009392801696735 |
|1.164087870627021 |1.180781251656718 |
|2.616811720667939 |2.257507129418043 |
|0.7677907235557111|0.7152298408076712|
|2.5565293934609676|2.2599159060634477|
+------------------+------------------+
only showing top 10 rows


📊 On original scale (kWh):


26/04/14 10:16:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 1

+------------------+------------------+
|actual_kwh        |pred_kwh          |
+------------------+------------------+
|7.91              |5.800462093316094 |
|2.4389999999999996|9.068331384897256 |
|3.2490000000000006|1.57750846759293  |
|4.042999999999999 |5.747105960828939 |
|9.553000000000003 |5.71839753830065  |
|4.548             |19.27508518955033 |
|2.203             |2.2569176815463474|
|12.692            |8.5592295175612   |
|1.1549999999999998|1.044656573774692 |
|11.890999999999998|8.582283320955861 |
+------------------+------------------+
only showing top 10 rows



26/04/14 10:16:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:16:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 1


📊 METRICS ON ORIGINAL SCALE (kWh):
  MAE: 2.7610 kWh
  RMSE: 4.0112 kWh

  Paper's XGBoost MAE: 2.6697 kWh
  Paper's XGBoost RMSE: 3.9451 kWh


In [ ]:
#  chay lai preprocessing r chay cai nay

In [58]:
print("="*70)
print("SUBSET TO PAPER'S 14,496 SESSIONS")
print("="*70)

# Dùng chính df_final (24,687 sessions)
print(f"Current sessions: {df_final.count():,}")

# Paper's exact fraction: 14496 / 24687 = 0.5872
target_count = 14496
fraction = target_count / df_final.count()

print(f"Target: {target_count:,} sessions")
print(f"Fraction: {fraction:.4f}")

# Random sample với seed=42 (như paper)
df_paper_size = df_final.sample(fraction=fraction, seed=42)

print(f"✅ After sampling: {df_paper_size.count():,} sessions")

# Kiểm tra phân bố
print("\n📊 Distribution check:")
print(f"  Unique years: {df_paper_size.select('year').distinct().orderBy('year').collect()}")
print(f"  Unique months: {df_paper_size.select('month').distinct().count()}")

# Train/test split (80/20, seed=42)
train_paper = df_paper_size.sample(fraction=0.8, seed=42)
test_paper = df_paper_size.subtract(train_paper)

print(f"\n📊 Train/test split:")
print(f"  Train: {train_paper.count():,} (expected: 11,597)")
print(f"  Test: {test_paper.count():,} (expected: 2,899)")

# Drop connectionTime_utc
train_paper = train_paper.drop("connectionTime_utc")
test_paper = test_paper.drop("connectionTime_utc")

SUBSET TO PAPER'S 14,496 SESSIONS


26/04/14 10:29:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:29:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Current sessions: 24,687


26/04/14 10:30:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

Target: 14,496 sessions
Fraction: 0.5872


26/04/14 10:30:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

✅ After sampling: 14,595 sessions

📊 Distribution check:


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`hour`, `season`, `month`, `day_of_year`, `duration`].;
'Project ['year]
+- Sample 0.0, 0.5871916393243407, false, 42
   +- Project [hour#22215, day_of_week#22224, month#22234, season#22284, duration#21635, charging_duration#21641, charging_duration_log#21648, hour_sin#22332, hour_cos#22350, day_of_year#22257, week_of_year#22270, is_holiday#22315, lag_1_log#22495, lag_2_log#22515, lag_3_log#22536, rolling_mean_3_log#22630, rolling_mean_5_log#22654, kWhDelivered_log#24410, connectionTime_utc#21513]
      +- Project [kWhDelivered#21400, connectionTime_utc#21513, disconnectTime_utc#21524, doneChargingTime_utc#21536, duration#21635, charging_duration#21641, charging_duration_log#21648, hour#22215, day_of_week#22224, month#22234, year#22245, day_of_year#22257, week_of_year#22270, season#22284, is_weekend#22299, is_holiday#22315, hour_

In [59]:
print("="*70)
print("SUBSET TO PAPER'S 14,496 SESSIONS (FIXED)")
print("="*70)

from pyspark.sql.functions import year

# df_final hiện tại có connectionTime_utc, có thể extract year lại
df_final_with_year = df_final.withColumn("year", year("connectionTime_utc"))

print(f"Current sessions: {df_final_with_year.count():,}")

# Paper's exact fraction: 14496 / 24687 = 0.5872
target_count = 14496
fraction = target_count / df_final_with_year.count()

print(f"Target: {target_count:,} sessions")
print(f"Fraction: {fraction:.4f}")

# Random sample với seed=42 (như paper)
df_paper_size = df_final_with_year.sample(fraction=fraction, seed=42)

print(f"✅ After sampling: {df_paper_size.count():,} sessions")

# Kiểm tra phân bố
print("\n📊 Distribution check:")
print(f"  Unique years: {df_paper_size.select('year').distinct().orderBy('year').collect()}")
print(f"  Unique months: {df_paper_size.select('month').distinct().count()}")

# Train/test split (80/20, seed=42, random)
train_paper = df_paper_size.sample(fraction=0.8, seed=42)
test_paper = df_paper_size.subtract(train_paper)

print(f"\n📊 Train/test split:")
print(f"  Train: {train_paper.count():,} (expected: ~11,597)")
print(f"  Test: {test_paper.count():,} (expected: ~2,899)")

# Drop connectionTime_utc (không dùng cho training)
train_paper = train_paper.drop("connectionTime_utc", "year")
test_paper = test_paper.drop("connectionTime_utc", "year")

print(f"\n✅ Final train features: {len(train_paper.columns)}")
print(f"✅ Final test features: {len(test_paper.columns)}")

SUBSET TO PAPER'S 14,496 SESSIONS (FIXED)


26/04/14 10:30:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Current sessions: 24,687


26/04/14 10:30:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:30:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Target: 14,496 sessions
Fraction: 0.5872


...

  Unique months: 12

📊 Train/test split:


26/04/14 10:31:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

  Train: 11,741 (expected: ~11,597)


26/04/14 10:31:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 1

  Test: 2,854 (expected: ~2,899)

✅ Final train features: 18
✅ Final test features: 18


In [60]:
print("="*70)
print("TRAIN GBT ON PAPER'S EXACT DATASET (14,496 sessions)")
print("="*70)

from pyspark.ml.regression import GBTRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Features (17 features)
feature_cols = [
    'hour', 'day_of_week', 'month', 'season',
    'duration', 'charging_duration', 'charging_duration_log',
    'hour_sin', 'hour_cos', 'day_of_year', 'week_of_year', 'is_holiday',
    'lag_1_log', 'lag_2_log', 'lag_3_log',
    'rolling_mean_3_log', 'rolling_mean_5_log'
]

# Pipeline
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
gbt = GBTRegressor(
    featuresCol="scaled_features",
    labelCol="kWhDelivered_log",
    maxDepth=10,
    maxIter=100,
    stepSize=0.1,
    seed=42
)

pipeline = Pipeline(stages=[assembler, scaler, gbt])

# Train
print("Training...")
model_paper = pipeline.fit(train_paper)

# Predict
predictions_paper = model_paper.transform(test_paper)

# Evaluate
from pyspark.sql.functions import exp, col, abs, avg, sqrt, pow

predictions_orig = predictions_paper \
    .withColumn("actual_kwh", exp(col("kWhDelivered_log")) - 1) \
    .withColumn("pred_kwh", exp(col("prediction")) - 1)

metrics = predictions_orig.select(
    avg(abs(col("actual_kwh") - col("pred_kwh"))).alias("mae"),
    sqrt(avg(pow(col("actual_kwh") - col("pred_kwh"), 2))).alias("rmse")
).collect()[0]

print(f"\n📊 RESULTS ON PAPER'S DATASET (14,496 sessions):")
print(f"  MAE: {metrics['mae']:.4f} kWh")
print(f"  RMSE: {metrics['rmse']:.4f} kWh")

print(f"\n📊 COMPARISON:")
print(f"  {'Metric':<15} {'Paper XGBoost':<18} {'Spark GBT (14k)':<18} {'Difference':<12}")
print("-" * 65)
print(f"  {'MAE':<15} {2.6697:<18} {metrics['mae']:<18.4f} {metrics['mae'] - 2.6697:<+12.4f}")
print(f"  {'RMSE':<15} {3.9451:<18} {metrics['rmse']:<18.4f} {metrics['rmse'] - 3.9451:<+12.4f}")

TRAIN GBT ON PAPER'S EXACT DATASET (14,496 sessions)
Training...


26/04/14 10:31:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 10:31:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 1


📊 RESULTS ON PAPER'S DATASET (14,496 sessions):
  MAE: 2.7801 kWh
  RMSE: 4.0703 kWh

📊 COMPARISON:
  Metric          Paper XGBoost      Spark GBT (14k)    Difference  
-----------------------------------------------------------------
  MAE             2.6697             2.7801             +0.1104     
  RMSE            3.9451             4.0703             +0.1252     


In [26]:
# 1. Paper đã loại bỏ những session nào?
# 2. Tại sao paper có 14,496 sessions mà chúng ta có 25,813?

# Kiểm tra từng điều kiện:
print("=== SO SÁNH VỚI PAPER ===")

# Điều kiện 1: userID không null
step1 = df_step9.filter(col("userID").isNotNull())
print(f"1. After removing NULL userID: {step1.count():,}")

# Điều kiện 2: duration hợp lý (có thể paper chỉ lấy duration trong khoảng nào đó)
step2 = step1.filter((col("duration_seconds") >= 3600) & (col("duration_seconds") <= 86400))
print(f"2. After duration 1-24h: {step2.count():,}")

# Điều kiện 3: kWhDelivered trong khoảng nào đó
step3 = step2.filter((col("kWhDelivered") >= 1) & (col("kWhDelivered") <= 25))
print(f"3. After kWhDelivered 1-25 kWh: {step3.count():,}")

# Điều kiện 4: chỉ giữ station có nhiều sessions
from pyspark.sql.functions import count
station_counts = step3.groupBy("stationID").agg(count("*").alias("cnt"))
valid_stations = station_counts.filter(col("cnt") >= 10).select("stationID")
step4 = step3.join(valid_stations, on="stationID", how="inner")
print(f"4. After keeping stations with >=10 sessions: {step4.count():,}")

=== SO SÁNH VỚI PAPER ===


1. After removing NULL userID: 12,714


2. After duration 1-24h: 11,914


3. After kWhDelivered 1-25 kWh: 11,705


[Stage 449:==============================================>        (29 + 4) / 34]

4. After keeping stations with >=10 sessions: 11,685


In [27]:
print("="*70)
print("FINDING PAPER'S EXACT FILTERS")
print("="*70)

from pyspark.sql.functions import count, col, when

# Bắt đầu từ data gốc đã qua preprocessing (df_step9)
df_test = df_step9

print(f"Base data: {df_test.count():,}")

# Filter 1: userID not null (đã làm)
df_test = df_test.filter(col("userID").isNotNull())
print(f"1. userID not null: {df_test.count():,}")

# Filter 2: duration between 1-24 hours (đã làm)
df_test = df_test.filter((col("duration_seconds") >= 3600) & (col("duration_seconds") <= 86400))
print(f"2. duration 1-24h: {df_test.count():,}")

# Filter 3: kWhDelivered between 1-25 kWh (đã làm)
df_test = df_test.filter((col("kWhDelivered") >= 1) & (col("kWhDelivered") <= 25))
print(f"3. kWhDelivered 1-25: {df_test.count():,}")

# Filter 4: stations with >=10 sessions (đã làm)
station_counts = df_test.groupBy("stationID").agg(count("*").alias("cnt"))
valid_stations = station_counts.filter(col("cnt") >= 10).select("stationID")
df_test = df_test.join(valid_stations, on="stationID", how="inner")
print(f"4. stations >=10 sessions: {df_test.count():,}")

# Thử thêm các điều kiện khác:
print("\n=== THỬ CÁC ĐIỀU KIỆN BỔ SUNG ===")

# 5. Có thể paper chỉ lấy data đến 2020?
df_temp = df_test.filter(year(col("connectionTime_utc")) <= 2020)
print(f"5. Year <= 2020: {df_temp.count():,}")

# 6. Hoặc chỉ lấy từ 2019 trở đi?
df_temp = df_test.filter(year(col("connectionTime_utc")) >= 2019)
print(f"6. Year >= 2019: {df_temp.count():,}")

# 7. Loại bỏ charging_duration quá ngắn (< 5 phút)
df_temp = df_test.filter(col("charging_duration_seconds") >= 300)
print(f"7. charging_duration >= 5min: {df_temp.count():,}")

# 8. Chỉ lấy các session có doneChargingTime (đã sạc xong)
df_temp = df_test.filter(col("doneChargingTime_utc").isNotNull())
print(f"8. doneChargingTime not null: {df_temp.count():,}")

# 9. Kết hợp nhiều điều kiện
df_temp = df_test.filter(
    (year(col("connectionTime_utc")) >= 2019) &
    (col("charging_duration_seconds") >= 300) &
    (col("doneChargingTime_utc").isNotNull())
)
print(f"9. Combined (year>=2019 + duration>=5min + done not null): {df_temp.count():,}")

# 10. Kiểm tra phân bố theo month
print("\n📊 Distribution by month (after current filters):")
df_test.groupBy("month").count().orderBy("month").show()

# 11. Kiểm tra phân bố theo year
print("\n📊 Distribution by year:")
df_test.groupBy("year").count().orderBy("year").show()

FINDING PAPER'S EXACT FILTERS


Base data: 25,813


1. userID not null: 12,714


2. duration 1-24h: 11,914


3. kWhDelivered 1-25: 11,705


...

8. doneChargingTime not null: 11,685


9. Combined (year>=2019 + duration>=5min + done not null): 9,827

📊 Distribution by month (after current filters):


+-----+-----+
|month|count|
+-----+-----+
|    1| 1254|
|    2| 1096|
|    3| 1056|
|    4|  863|
|    5|  871|
|    6|  519|
|    7|  866|
|    8|  901|
|    9|  856|
|   10|  928|
|   11| 1340|
|   12| 1135|
+-----+-----+


📊 Distribution by year:


[Stage 521:===================================================>   (32 + 2) / 34]

+----+-----+
|year|count|
+----+-----+
|2018| 1858|
|2019| 7174|
|2020| 1338|
|2021| 1315|
+----+-----+



In [28]:
print("="*70)
print("GIẢ THUYẾT: PAPER KHÔNG LỌC NULL USERID")
print("="*70)

# Bắt đầu lại từ df_step9 (25,813 sessions)
df_hypothesis = df_step9

print(f"Base data (after outlier removal & missing values): {df_hypothesis.count():,}")

# Filter: duration 1-24h (KHÔNG lọc userID)
df_hypothesis = df_hypothesis.filter(
    (col("duration_seconds") >= 3600) & 
    (col("duration_seconds") <= 86400)
)
print(f"After duration 1-24h: {df_hypothesis.count():,}")

# Filter: kWhDelivered 1-25 kWh
df_hypothesis = df_hypothesis.filter(
    (col("kWhDelivered") >= 1) & 
    (col("kWhDelivered") <= 25)
)
print(f"After kWhDelivered 1-25: {df_hypothesis.count():,}")

# Filter: stations with >=10 sessions
station_counts = df_hypothesis.groupBy("stationID").agg(count("*").alias("cnt"))
valid_stations = station_counts.filter(col("cnt") >= 10).select("stationID")
df_hypothesis = df_hypothesis.join(valid_stations, on="stationID", how="inner")
print(f"After stations >=10 sessions: {df_hypothesis.count():,}")

print(f"\n📊 Result: {df_hypothesis.count():,} sessions")
print(f"Paper's count: 14,496")
print(f"Difference: {df_hypothesis.count() - 14496:,}")

# Kiểm tra phân bố userID null trong kết quả này
null_userid_count = df_hypothesis.filter(col("userID").isNull()).count()
print(f"\nSessions with NULL userID in this set: {null_userid_count:,}")
print(f"Non-NULL userID sessions: {df_hypothesis.count() - null_userid_count:,}")

# Phân bố theo year
print("\n📊 Distribution by year (without userID filter):")
df_hypothesis.groupBy("year").count().orderBy("year").show()

GIẢ THUYẾT: PAPER KHÔNG LỌC NULL USERID


Base data (after outlier removal & missing values): 25,813


After duration 1-24h: 23,082


After kWhDelivered 1-25: 21,286


After stations >=10 sessions: 21,277


...

Difference: 6,781



Sessions with NULL userID in this set: 9,573


Non-NULL userID sessions: 11,704

📊 Distribution by year (without userID filter):


[Stage 575:===================================================>   (32 + 2) / 34]

+----+-----+
|year|count|
+----+-----+
|2018|11248|
|2019| 7333|
|2020| 1368|
|2021| 1328|
+----+-----+



In [29]:
print("="*70)
print("THỬ CÁC COMBINATION KHÁC")
print("="*70)

# Option A: Chỉ lọc duration + kWhDelivered + stations (không userID)
df_option_a = df_step9 \
    .filter((col("duration_seconds") >= 3600) & (col("duration_seconds") <= 86400)) \
    .filter((col("kWhDelivered") >= 1) & (col("kWhDelivered") <= 25))

station_counts = df_option_a.groupBy("stationID").agg(count("*").alias("cnt"))
valid_stations = station_counts.filter(col("cnt") >= 10).select("stationID")
df_option_a = df_option_a.join(valid_stations, on="stationID", how="inner")
print(f"Option A (no userID filter): {df_option_a.count():,}")

# Option B: Thêm filter year >= 2019
df_option_b = df_option_a.filter(year(col("connectionTime_utc")) >= 2019)
print(f"Option B (year >= 2019): {df_option_b.count():,}")

# Option C: Thêm filter charging_duration >= 5min
df_option_c = df_option_a.filter(col("charging_duration_seconds") >= 300)
print(f"Option C (charging_duration >= 5min): {df_option_c.count():,}")

# Option D: Random sample với seed=42 để đúng 14,496
if df_option_a.count() > 14496:
    fraction = 14496 / df_option_a.count()
    df_option_d = df_option_a.sample(fraction=fraction, seed=42)
    print(f"Option D (random sample {fraction:.3f}): {df_option_d.count():,}")

THỬ CÁC COMBINATION KHÁC


Option A (no userID filter): 21,277


Option B (year >= 2019): 10,029


Option C (charging_duration >= 5min): 21,271


[Stage 617:===================================================>   (32 + 2) / 34]

Option D (random sample 0.681): 14,597


In [30]:
print("="*70)
print("✅ PAPER'S EXACT PREPROCESSING FORMULA")
print("="*70)

# Bước 1: Áp dụng các bộ lọc cơ bản (KHÔNG lọc userID)
df_paper = df_step9 \
    .filter((col("duration_seconds") >= 3600) & (col("duration_seconds") <= 86400)) \
    .filter((col("kWhDelivered") >= 1) & (col("kWhDelivered") <= 25))

# Bước 2: Chỉ giữ stations có >= 10 sessions
station_counts = df_paper.groupBy("stationID").agg(count("*").alias("cnt"))
valid_stations = station_counts.filter(col("cnt") >= 10).select("stationID")
df_paper = df_paper.join(valid_stations, on="stationID", how="inner")

print(f"After filters: {df_paper.count():,} sessions")

# Bước 3: Random sample để đúng 14,496 (seed=42)
target_count = 14496
fraction = target_count / df_paper.count()
df_paper_final = df_paper.sample(fraction=fraction, seed=42)

print(f"After random sample (fraction={fraction:.4f}, seed=42): {df_paper_final.count():,} sessions")
print(f"Paper's target: 14,496")
print(f"Match: {'✅ YES' if df_paper_final.count() == 14496 else '⚠️ Close'}")

# Lưu lại kết quả đúng như paper
print("\n📊 Final paper-matched dataset:")
print(f"  Total sessions: {df_paper_final.count():,}")
print(f"  Features: {len(df_paper_final.columns)}")
print(f"  Time range:")
df_paper_final.select(min("connectionTime_utc"), max("connectionTime_utc")).show()

✅ PAPER'S EXACT PREPROCESSING FORMULA


After filters: 21,277 sessions


After random sample (fraction=0.6813, seed=42): 14,597 sessions
Paper's target: 14,496


Match: ⚠️ Close

📊 Final paper-matched dataset:


  Total sessions: 14,597
  Features: 30
  Time range:


[Stage 659:===================================================>   (32 + 2) / 34]

+-----------------------+-----------------------+
|min(connectionTime_utc)|max(connectionTime_utc)|
+-----------------------+-----------------------+
|    2018-04-25 13:45:10|    2021-09-13 23:11:12|
+-----------------------+-----------------------+



In [31]:
print("="*70)
print("FINAL TRAIN/TEST SPLIT (MATCH PAPER)")
print("="*70)

# Sắp xếp theo thời gian và chia 80/20 (không xáo trộn)
df_ordered = df_paper_final.orderBy("connectionTime_utc")

total_rows = df_ordered.count()
train_size = int(total_rows * 0.8)
test_size = total_rows - train_size

print(f"Total: {total_rows:,}")
print(f"Train (80%): {train_size:,}")
print(f"Test (20%): {test_size:,}")

# Đánh index theo thời gian
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

window_spec = Window.orderBy("connectionTime_utc")
df_with_index = df_ordered.withColumn("row_num", row_number().over(window_spec))

# Chia train/test
train_final = df_with_index.filter(col("row_num") <= train_size).drop("row_num")
test_final = df_with_index.filter(col("row_num") > train_size).drop("row_num")

print(f"\n✅ Train: {train_final.count():,} rows")
print(f"✅ Test: {test_final.count():,} rows")

# Kiểm tra time range
print("\n📅 Train time range:")
train_final.select(min("connectionTime_utc"), max("connectionTime_utc")).show()
print("📅 Test time range:")
test_final.select(min("connectionTime_utc"), max("connectionTime_utc")).show()

FINAL TRAIN/TEST SPLIT (MATCH PAPER)


Total: 14,597
Train (80%): 11,677
Test (20%): 2,920


26/04/14 09:23:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 0


✅ Train: 11,677 rows


26/04/14 09:23:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 0

✅ Test: 2,920 rows

📅 Train time range:


26/04/14 09:23:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:23:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 0

+-----------------------+-----------------------+
|min(connectionTime_utc)|max(connectionTime_utc)|
+-----------------------+-----------------------+
|    2018-04-25 13:45:10|    2019-09-30 16:51:06|
+-----------------------+-----------------------+

📅 Test time range:


26/04/14 09:24:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:24:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:24:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:24:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:24:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:24:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 0

+-----------------------+-----------------------+
|min(connectionTime_utc)|max(connectionTime_utc)|
+-----------------------+-----------------------+
|    2019-09-30 16:58:38|    2021-09-13 23:11:12|
+-----------------------+-----------------------+



In [32]:
print("="*70)
print("TRAIN GBT REGRESSOR ON SPARK ML")
print("="*70)

from pyspark.ml.regression import GBTRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
import numpy as np
import pandas as pd

# 1. Chuẩn bị features và target
print("\n📊 PREPARING FEATURES...")

# Chọn features (dựa trên paper)
feature_cols = [
    'hour', 'day_of_week', 'month', 'is_weekend', 'season',
    'hour_sin', 'hour_cos', 
    'duration_seconds', 'charging_duration_log',
    'lag_1_kWhDelivered', 'lag_2_kWhDelivered', 
    'rolling_mean_7_kWh', 'rolling_std_7_kWh', 'rolling_mean_duration'
]

# Target là kWhDelivered_log (đã log transform)
target_col = 'kWhDelivered_log'

print(f"  Features: {len(feature_cols)} columns")
print(f"  Features list: {feature_cols}")
print(f"  Target: {target_col}")

# 2. Xây dựng pipeline
print("\n🔧 BUILDING PIPELINE...")

# Vector Assembler: gộp features thành vector
assembler = VectorAssembler(
    inputCols=feature_cols, 
    outputCol="features",
    handleInvalid="skip"
)

# StandardScaler: chuẩn hóa features (mean=0, std=1)
scaler = StandardScaler(
    inputCol="features", 
    outputCol="scaled_features",
    withStd=True, 
    withMean=True
)

# GBT Regressor (hyperparameters từ paper XGBoost)
gbt = GBTRegressor(
    featuresCol="scaled_features",
    labelCol=target_col,
    maxDepth=10,          # Tương tự XGBoost
    maxIter=100,          # Số cây (n_estimators)
    stepSize=0.1,         # Learning rate
    subsamplingRate=0.8,  # Sample 80% data mỗi cây
    minInstancesPerNode=5,
    seed=42,
    cacheNodeIds=False,   # Giảm memory cho 4GB RAM
    maxMemoryInMB=256     # Giới hạn memory cho histogram
)

# Pipeline
pipeline = Pipeline(stages=[assembler, scaler, gbt])

print("  ✅ Pipeline created")

# 3. Train model
print("\n🏋️ TRAINING GBT MODEL...")
import time
start_time = time.time()

model = pipeline.fit(train_final)

train_time = time.time() - start_time
print(f"  ✅ Training completed in {train_time:.2f} seconds")

# 4. Predictions
print("\n📈 MAKING PREDICTIONS...")
predictions = model.transform(test_final)

# 5. Evaluate on log scale (original paper metric)
print("\n📊 EVALUATION ON LOG SCALE:")
evaluator_rmse = RegressionEvaluator(
    labelCol=target_col, 
    predictionCol="prediction", 
    metricName="rmse"
)
evaluator_mae = RegressionEvaluator(
    labelCol=target_col, 
    predictionCol="prediction", 
    metricName="mae"
)
evaluator_r2 = RegressionEvaluator(
    labelCol=target_col, 
    predictionCol="prediction", 
    metricName="r2"
)

rmse_log = evaluator_rmse.evaluate(predictions)
mae_log = evaluator_mae.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"  RMSE (log scale): {rmse_log:.4f}")
print(f"  MAE (log scale): {mae_log:.4f}")
print(f"  R²: {r2:.4f}")

# 6. Convert back to original scale (kWh)
print("\n📊 EVALUATION ON ORIGINAL SCALE (kWh):")

# Hàm inverse log transform: exp(x) - 1
from pyspark.sql.functions import exp, col

predictions_with_original = predictions \
    .withColumn("kWhDelivered_pred", exp(col("prediction")) - 1) \
    .withColumn("kWhDelivered_actual", exp(col(target_col)) - 1)

# Tính RMSE/MAE trên original scale
from pyspark.sql.functions import sqrt, avg, pow, abs

metrics = predictions_with_original.select(
    sqrt(avg(pow(col("kWhDelivered_actual") - col("kWhDelivered_pred"), 2))).alias("rmse_original"),
    avg(abs(col("kWhDelivered_actual") - col("kWhDelivered_pred"))).alias("mae_original"),
    (1 - avg(pow(col("kWhDelivered_actual") - col("kWhDelivered_pred"), 2)) / 
     avg(pow(col("kWhDelivered_actual") - avg(col("kWhDelivered_actual")).over(), 2))).alias("r2_original")
).collect()[0]

print(f"  RMSE (kWh): {metrics['rmse_original']:.4f}")
print(f"  MAE (kWh): {metrics['mae_original']:.4f}")
print(f"  R²: {metrics['r2_original']:.4f}")

# 7. So sánh với paper's XGBoost baseline
print("\n" + "="*70)
print("📊 COMPARISON WITH PAPER'S XGBOOST BASELINE")
print("="*70)

# Paper's results (từ bảng trong paper)
paper_rmse_kwh = 3.42  # Giả định - thay bằng số thực từ paper
paper_mae_kwh = 2.15   # Giả định - thay bằng số thực từ paper
paper_r2 = 0.62        # Giả định - thay bằng số thực từ paper

print(f"\n{'Metric':<20} {'Spark GBT':<15} {'Paper XGBoost':<15} {'Difference':<15}")
print("-"*65)
print(f"{'RMSE (kWh)':<20} {metrics['rmse_original']:<15.4f} {paper_rmse_kwh:<15.4f} {metrics['rmse_original'] - paper_rmse_kwh:<+15.4f}")
print(f"{'MAE (kWh)':<20} {metrics['mae_original']:<15.4f} {paper_mae_kwh:<15.4f} {metrics['mae_original'] - paper_mae_kwh:<+15.4f}")
print(f"{'R²':<20} {metrics['r2_original']:<15.4f} {paper_r2:<15.4f} {metrics['r2_original'] - paper_r2:<+15.4f}")

# 8. Feature Importance
print("\n🔍 FEATURE IMPORTANCE ANALYSIS:")
feature_importance = []
for i, feature in enumerate(feature_cols):
    importance = model.stages[-1].featureImportances[i]
    feature_importance.append((feature, importance))

feature_importance.sort(key=lambda x: x[1], reverse=True)

print("\nTop 10 most important features:")
for i, (feature, importance) in enumerate(feature_importance[:10], 1):
    print(f"  {i:2}. {feature:<25} {importance:.4f}")

# 9. Lưu model
print("\n💾 SAVING MODEL...")
model_path = "models/spark_gbt_model"
model.write().overwrite().save(model_path)
print(f"  ✅ Model saved to: {model_path}")

# 10. Sample predictions
print("\n📋 SAMPLE PREDICTIONS (first 10 test rows):")
predictions_with_original.select(
    "kWhDelivered_actual", "kWhDelivered_pred", 
    "hour", "stationID"
).show(10, truncate=False)

TRAIN GBT REGRESSOR ON SPARK ML

📊 PREPARING FEATURES...
  Features: 14 columns
  Features list: ['hour', 'day_of_week', 'month', 'is_weekend', 'season', 'hour_sin', 'hour_cos', 'duration_seconds', 'charging_duration_log', 'lag_1_kWhDelivered', 'lag_2_kWhDelivered', 'rolling_mean_7_kWh', 'rolling_std_7_kWh', 'rolling_mean_duration']
  Target: kWhDelivered_log

🔧 BUILDING PIPELINE...
  ✅ Pipeline created

🏋️ TRAINING GBT MODEL...


26/04/14 09:28:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:28:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:28:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:28:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:28:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:28:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 0

  ✅ Training completed in 321.00 seconds

📈 MAKING PREDICTIONS...

📊 EVALUATION ON LOG SCALE:


26/04/14 09:33:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:33:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:33:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:33:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:33:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 09:33:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/14 0

  RMSE (log scale): 0.4508
  MAE (log scale): 0.3492
  R²: 0.4198

📊 EVALUATION ON ORIGINAL SCALE (kWh):


TypeError: Column.over() missing 1 required positional argument: 'window'